# Reproducing the HOG paper


## Preparations

### dependence

In [ ]:
import os
import cv2
import re
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.svm import LinearSVC
from sklearn.svm import SVC  
from PIL import Image, ImageFile
import warnings
import pickle

### Paths and default parameters

In [ ]:
WIN_SIZE = (64, 128)

WIN_W, WIN_H = WIN_SIZE  # 64x128
WIN_ASPECT_RATIO = WIN_W / WIN_H  
PERSON_MARGIN_RATIO = 0.25 
INRIA_ROOT = r""  # Change this to the root directory of your INRIA dataset
TRAIN_POS_DIR = os.path.join("Train", "pos")
TRAIN_NEG_DIR = os.path.join("Train", "neg")
TEST_ANNO_DIR = os.path.join("Test", "annotations")
TEST_POS_DIR = os.path.join("Test", "pos")
TEST_NEG_DIR = os.path.join("Test", "neg")


CROP_X1, CROP_Y1 = (96 - WIN_SIZE[0]) // 2, (160 - WIN_SIZE[1]) // 2
CROP_X2, CROP_Y2 = CROP_X1 + WIN_SIZE[0], CROP_Y1 + WIN_SIZE[1]
NEG_SAMPLES_PER_IMG = 10

HOG_CELL_SIZE = (8, 8)          
HOG_BLOCK_SIZE = (16, 16)        
HOG_BLOCK_STRIDE = (8, 8)        
HOG_NBINS = 9                    
HOG_DERIV_APERTURE = 1              
HOG_WIN_SIGMA = 8.0                 
HOG_NORM_TYPE = cv2.HOGDESCRIPTOR_L2HYS  
HOG_L2_HYS_THRESH = 0.2          
HOG_GAMMA_CORRECTION = False        
HOG_NLEVELS = 64                    
HOG_SIGNED_GRADIENT = False         
HOG_DIM = 3780                   
SVM_C = 0.8
SVM_MAX_ITER = 10000

PYRAMID_SCALE = 1.05  
WINDOW_STRIDE = 8 
WINDOW_STRIDE_TUPLE = (WINDOW_STRIDE, WINDOW_STRIDE)  
SVM_BATCH_SIZE = 1024                           
NMS_THRESHOLD = 0.3                              

### Function definition


In [ ]:
def parse_inria_annotation(anno_path):
    bbox_list = []
    bbox_pattern = re.compile(
        r'Bounding box for object \d+.*\(Xmin, Ymin\) - \(Xmax, Ymax\) : \((\d+), (\d+)\) - \((\d+), (\d+)\)'
    )
    
    try:
        with open(anno_path, 'r', encoding='utf-8', errors='ignore') as f:
            anno_content = f.read()
        
        matches = bbox_pattern.findall(anno_content)
        for match in matches:
            Xmin = int(match[0])
            Ymin = int(match[1])
            Xmax = int(match[2])
            Ymax = int(match[3])
            if Xmax > Xmin and Ymax > Ymin:
                bbox_list.append([Xmin, Ymin, Xmax, Ymax])
    
    except Exception as e:
        print(f"Failed to parse the annotation file: {os.path.basename(anno_path)}, 错误: {str(e)}")
    
    return bbox_list

def crop_person_with_margin(img, bbox):
    img_h, img_w = img.shape
    Xmin, Ymin, Xmax, Ymax = bbox

    person_w = Xmax - Xmin
    person_h = Ymax - Ymin

    margin_w = int(person_w * PERSON_MARGIN_RATIO)
    margin_h = int(person_h * PERSON_MARGIN_RATIO)
    new_x1 = max(0, Xmin - margin_w)
    new_y1 = max(0, Ymin - margin_h)
    new_x2 = min(img_w, Xmax + margin_w)
    new_y2 = min(img_h, Ymax + margin_h)

    crop_w = new_x2 - new_x1
    crop_h = new_y2 - new_y1
    current_ratio = crop_w / crop_h

    if current_ratio > WIN_ASPECT_RATIO:
        new_crop_h = int(crop_w / WIN_ASPECT_RATIO)
        delta_h = new_crop_h - crop_h
        new_y1 = max(0, new_y1 - delta_h // 2)
        new_y2 = min(img_h, new_y2 + delta_h - delta_h // 2)
    else:
        new_crop_w = int(crop_h * WIN_ASPECT_RATIO)
        delta_w = new_crop_w - crop_w
        new_x1 = max(0, new_x1 - delta_w // 2)
        new_x2 = min(img_w, new_x2 + delta_w - delta_w // 2)

    img_cropped = img[new_y1:new_y2, new_x1:new_x2]
    img_resized = cv2.resize(img_cropped, WIN_SIZE, interpolation=cv2.INTER_LINEAR)
    
    return img_resized

In [ ]:
def safe_read_image(img_path, to_gray=False):
    try:
        with Image.open(img_path) as img:
            img.info.pop("icc_profile", None)
            img.info.pop("exif", None)
            if to_gray:
                img = img.convert("L")
            img_np = np.array(img)
        return img_np
    except Exception as e:
        print(f"Skip damaged images: {os.path.basename(img_path)}")
        return None

def pyramid(img):
    scale = 1.0
    while True:
        scaled_img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
        if scaled_img.shape[0] < WIN_H or scaled_img.shape[1] < WIN_W:
            break
        yield scaled_img
        scale /= PYRAMID_SCALE

def nms(detections, thresh):
    if len(detections) == 0:
        return []
    
    # detections: [x, y, w, h, score]
    x1 = detections[:, 0]
    y1 = detections[:, 1]
    x2 = detections[:, 0] + detections[:, 2]
    y2 = detections[:, 1] + detections[:, 3]
    scores = detections[:, 4]
    
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0.0, xx2 - xx1 + 1)
        h = np.maximum(0.0, yy2 - yy1 + 1)
        inter = w * h
        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        
        inds = np.where(ovr <= thresh)[0]
        order = order[inds + 1]
    
    return keep

def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xx1 = max(x1, x2)
    yy1 = max(y1, y2)
    xx2 = min(x1 + w1, x2 + w2)
    yy2 = min(y1 + h1, y2 + h2)
    
    inter = max(0, xx2 - xx1) * max(0, yy2 - yy1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0

def match_detections_to_gt(detections, gt_bboxes, iou_thresh=0.5):
    matched = set()
    for det in detections:
        max_iou = 0
        max_idx = -1
        for idx, gt in enumerate(gt_bboxes):
            if idx in matched:
                continue
            iou = compute_iou(det[:4], gt)
            if iou > max_iou:
                max_iou = iou
                max_idx = idx
        if max_iou >= iou_thresh:
            matched.add(max_idx)
    return len(matched)

In [ ]:
def get_test_det_results(pos_img_dir, anno_dir, neg_dir, hog_detector, svm_model, HOG_DIM):
    all_scores = []
    all_labels = []
    total_test_windows = 0  

    print("📥 Processing test positive samples (analysing, annotating and cropping pedestrians)...")
    pos_img_list = [f for f in os.listdir(pos_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(pos_img_list):
        img_path = os.path.join(pos_img_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        img_prefix = os.path.splitext(img_name)[0]
        anno_path = os.path.join(anno_dir, f"{img_prefix}.txt")
        if not os.path.exists(anno_path):
            print(f"Skip images without captions: {img_name}")
            continue
        
        bbox_list = parse_inria_annotation(anno_path)
        if len(bbox_list) == 0:
            print(f"Skip images without valid pedestrian annotations: {img_name}")
            continue
        
        for bbox in bbox_list:
            try:
                person_img = crop_person_with_margin(img, bbox)
                feature = hog_detector.compute(person_img).flatten().reshape(1, -1)
                score = svm_model.decision_function(feature)[0]+1.5
                all_scores.append(score)
                all_labels.append(1)
            except Exception as e:
                print(f"Failed to crop the pedestrian: {img_name}, bbox={bbox}, Error: {str(e)}")
                continue

    print("📥 Handling negative test samples (exhaustive sliding window search)...")
    neg_img_list = [f for f in os.listdir(neg_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(neg_img_list):
        img_path = os.path.join(neg_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        
        for scale_img in pyramid(img):
            img_h, img_w = scale_img.shape
            x_coords = np.arange(0, img_w - WIN_W + 1, WINDOW_STRIDE)
            y_coords = np.arange(0, img_h - WIN_H + 1, WINDOW_STRIDE)
            
            if len(x_coords) == 0 or len(y_coords) == 0:
                continue
            
            num_windows = len(x_coords) * len(y_coords)
            total_test_windows += num_windows
            
            batch_features = np.zeros((num_windows, HOG_DIM), dtype=np.float32)
            window_idx = 0
            for y in y_coords:
                for x in x_coords:
                    window = scale_img[y:y+WIN_H, x:x+WIN_W]
                    batch_features[window_idx] = hog_detector.compute(window).flatten()
                    window_idx += 1
            
            batch_scores = svm_model.decision_function(batch_features)
            all_scores.extend(batch_scores)
            all_labels.extend([-1] * len(batch_scores))

    return np.array(all_scores), np.array(all_labels), total_test_windows

In [ ]:
def load_positive_samples(pos_dir):
    pos_images = []
    pos_img_list = [f for f in os.listdir(pos_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(pos_img_list, desc="Loading positive samples"):
        img_path = os.path.join(pos_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        if img.shape[0] < 160 or img.shape[1] < 96:
            continue
        img_cropped = img[CROP_Y1:CROP_Y2, CROP_X1:CROP_X2]
        if img_cropped.shape != (WIN_H, WIN_W):
            continue
        pos_images.append(img_cropped)
    
    return pos_images

def load_negative_samples(neg_dir, samples_per_img=NEG_SAMPLES_PER_IMG):
    neg_images = []
    neg_img_list = [f for f in os.listdir(neg_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(neg_img_list, desc="Load initial negative samples"):
        img_path = os.path.join(neg_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        h, w = img.shape
        for _ in range(samples_per_img):
            x = np.random.randint(0, w - WIN_W + 1)
            y = np.random.randint(0, h - WIN_H + 1)
            patch = img[y:y + WIN_H, x:x + WIN_W]
            neg_images.append(patch)
    
    return neg_images

In [ ]:
def extract_hog_features(images, hog):
    features = []
    for img in tqdm(images, desc="Extract HOG features"):
        if img.shape != (WIN_H, WIN_W):
            continue
        feature = hog.compute(img)
        features.append(feature.flatten())
    return np.array(features)

In [ ]:
def compute_det_curve(scores, labels, total_windows):
    sorted_idx = np.argsort(scores)[::-1]
    sorted_scores = scores[sorted_idx]
    sorted_labels = labels[sorted_idx]
    tp = np.cumsum(sorted_labels == 1)
    fp = np.cumsum(sorted_labels == -1)
    total_pos = np.sum(labels == 1)
    miss_rate = 1 - (tp / total_pos)
    fppw = fp / total_windows

    return miss_rate, fppw

### Loading training samples

In [ ]:
pos_images = load_positive_samples(TRAIN_POS_DIR)
print(f"Sample loading complete! Number of valid samples：{len(pos_images)}")

neg_images = load_negative_samples(TRAIN_NEG_DIR)
print(f"Initial negative samples have been loaded! Number of valid samples：{len(neg_images)}")

### Difficult loading examples

In [ ]:
hard_data = np.load("hard_examples_60k.npy", allow_pickle=True).item()

hard_imgs = hard_data["images"] 

hard_imgs_gray = np.array([np.array(Image.fromarray(img).convert('L')) for img in hard_imgs])

print(f"neg_images shape: {np.array(neg_images).shape}")
print(f"hard_imgs_gray shape: {hard_imgs_gray.shape}")

neg_images_with_hard = list(neg_images) + list(hard_imgs_gray)

print(f"Number of original neg_images: {len(neg_images)}")
print(f"Number of hard_imgs_gray: {hard_imgs_gray.shape[0]}")
print(f"Total number after consolidation: {len(neg_images_with_hard)}")
print(f"Combined data type: {type(neg_images_with_hard)}")
print(f"The first sample type and shape: {type(neg_images_with_hard[0])}, {neg_images_with_hard[0].shape}")

## Parametric experiments

### 6.1 Gamma/Colour Normalization

In [ ]:
param_combos_61 = [
    {'gamma_correction': False, 'name': 'no gamma_correction'},
    {'gamma_correction': True, 'name': 'with gamma_correction (square root)'},
]

In [ ]:
results_61 = []

for params in param_combos_61:
    print(f"\n🔧 Test 6.1 Gamma: {params['name']}")
    
    hog = cv2.HOGDescriptor(
        WIN_SIZE, HOG_BLOCK_SIZE, HOG_BLOCK_STRIDE, HOG_CELL_SIZE, HOG_NBINS,
        HOG_DERIV_APERTURE, HOG_WIN_SIGMA, HOG_NORM_TYPE, HOG_L2_HYS_THRESH,
        params['gamma_correction'], HOG_NLEVELS, HOG_SIGNED_GRADIENT
    )
    
    X_pos = extract_hog_features(pos_images, hog)
    X_neg = extract_hog_features(neg_images_with_hard, hog)
    
    y_pos = np.ones(X_pos.shape[0])
    y_neg = -np.ones(X_neg.shape[0])
    X_train = np.vstack((X_pos, X_neg))
    y_train = np.hstack((y_pos, y_neg))
    
    svm = LinearSVC(C=SVM_C, max_iter=SVM_MAX_ITER, dual=False, random_state=42)
    svm.fit(X_train, y_train)
    print("SVM training complete！")
    
    test_scores, test_labels, total_test_windows = get_test_det_results(
        TEST_POS_DIR, TEST_ANNO_DIR, TEST_NEG_DIR, hog, svm, hog.getDescriptorSize()
    )
    print(f"✅ Test complete! Total number of windows：{total_test_windows}")
    
    miss_rate, fppw = compute_det_curve(test_scores, test_labels, total_test_windows)
    closest_idx = np.argmin(np.abs(fppw - 1e-4))
    print(f"The false negative rate when FPPW ≈ 1e-4: {miss_rate[closest_idx]:.2%}")
    
    results_61.append({
        'params': params,
        'miss_rate': miss_rate,
        'fppw': fppw
    })

In [ ]:
colors = ['blue', 'red']
plt.figure(figsize=(12, 8))
for i, result in enumerate(results_61):
    label = result['params']['name']
    plt.plot(result['fppw'], result['miss_rate'], label=label, color=colors[i])

plt.axvline(x=1e-4, color='red', linestyle='--', linewidth=1, label='FPPW=1e-4')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('FPPW')
plt.ylabel('Miss Rate')
plt.title('6.1 HOG with Different Gamma Correction DET Curve')
plt.ylim(1e-2, 0.5)
plt.yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])

from matplotlib.ticker import FormatStrFormatter
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
with open('results_61.pkl', 'wb') as f:
    pickle.dump(results_61, f)
plt.legend()
plt.grid(True)
plt.show()

### 6.2 Gradient Computation

In [ ]:
def extract_hog_with_sigma(images, hog, sigma):
    features = []
    for img in tqdm(images, desc=f"提取HOG (σ={sigma})"):
        if sigma > 0:
            blurred = cv2.GaussianBlur(img, (0, 0), sigma)
        else:
            blurred = img.copy()
        if blurred.shape != (WIN_H, WIN_W):
            continue
        feature = hog.compute(blurred).flatten()
        features.append(feature)
    return np.array(features)

param_combos_62 = [
    {'sigma': 0.0, 'name': 'σ=0 (no smoothing)'},
    {'sigma': 0.5, 'name': 'σ=0.5'},
    {'sigma': 1.0, 'name': 'σ=1'},
    {'sigma': 2.0, 'name': 'σ=2'},
    {'sigma': 3.0, 'name': 'σ=3'},
]

In [ ]:
results_62 = []

for params in param_combos_62:
    print(f"\n🔧 Test 6.2: Gradient Scale σ={params['sigma']}")
    
    hog = cv2.HOGDescriptor(
        WIN_SIZE, HOG_BLOCK_SIZE, HOG_BLOCK_STRIDE, HOG_CELL_SIZE, HOG_NBINS,
        1,  # derivAperture=1（论文[-1,0,1]）
        HOG_WIN_SIGMA, HOG_NORM_TYPE, HOG_L2_HYS_THRESH,
        HOG_GAMMA_CORRECTION, HOG_NLEVELS, HOG_SIGNED_GRADIENT
    )
    
    hog_dim = hog.getDescriptorSize()
    print(f"HOG feature dimensions: {hog_dim}（The default plagiarism checker for the thesis is 3780）")

    X_pos = extract_hog_with_sigma(pos_images, hog, params['sigma'])
    X_neg = extract_hog_with_sigma(neg_images_with_hard, hog, params['sigma'])
    
    y_pos = np.ones(X_pos.shape[0])
    y_neg = -np.ones(X_neg.shape[0])
    X_train = np.vstack((X_pos, X_neg))
    y_train = np.hstack((y_pos, y_neg))
    
    svm = LinearSVC(C=SVM_C, max_iter=SVM_MAX_ITER, dual=False, random_state=42)
    svm.fit(X_train, y_train)
    print("SVM training complete！")
    
    # 测试（完全复制6.3节）
    test_scores, test_labels, total_test_windows = get_test_det_results(
        TEST_POS_DIR, TEST_ANNO_DIR, TEST_NEG_DIR, hog, svm, hog_dim
    )
    print(f"✅ Test complete! Total number of windows：{total_test_windows}")
    
    miss_rate, fppw = compute_det_curve(test_scores, test_labels, total_test_windows)
    closest_idx = np.argmin(np.abs(fppw - 1e-4))
    print(f"The false negative rate when FPPW ≈ 1e-4: {miss_rate[closest_idx]:.2%}")
    
    results_62.append({
        'params': params,
        'miss_rate': miss_rate,
        'fppw': fppw
    })

In [ ]:
colors = ['blue', 'red', 'green', 'orange', 'purple']
plt.figure(figsize=(12, 8))
for i, result in enumerate(results_62):
    label = result['params']['name']
    plt.plot(result['fppw'], result['miss_rate'], label=label, color=colors[i])

plt.axvline(x=1e-4, color='red', linestyle='--', linewidth=1, label='FPPW=1e-4')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('FPPW')
plt.ylabel('Miss Rate')
plt.title('6.2 HOG with Different Gaussian Smoothing DET Curve')

plt.ylim(1e-2, 0.5)
plt.yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])

from matplotlib.ticker import FormatStrFormatter
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
with open('results_62.pkl', 'wb') as f:
    pickle.dump(results_62, f)
plt.legend()
plt.grid(True)
plt.show()

### 6.3 Spatial / Orientation Binning

In [ ]:
param_combos_63 = [
    {'nbins': 9, 'signed': False, 'name': 'nbins=9, unsigned'},
    {'nbins': 6, 'signed': False, 'name': 'nbins=6, unsigned'},
    {'nbins': 4, 'signed': False, 'name': 'nbins=4, unsigned'},
    {'nbins': 3, 'signed': False, 'name': 'nbins=3, unsigned'},
    {'nbins': 18, 'signed': True, 'name': 'nbins=18, signed'},
    {'nbins': 12, 'signed': True, 'name': 'nbins=12, signed'},
    {'nbins': 8, 'signed': True, 'name': 'nbins=8, signed'},
    {'nbins': 6, 'signed': True, 'name': 'nbins=6, signed'},
]

In [ ]:
results_63 = []

for params in param_combos_63:
    print(f"\n🔧 Test parameter combinations: nbins={params['nbins']}, signed={params['signed']}")

    hog = cv2.HOGDescriptor(
    WIN_SIZE, HOG_BLOCK_SIZE, HOG_BLOCK_STRIDE, HOG_CELL_SIZE, params['nbins'],
    HOG_DERIV_APERTURE, HOG_WIN_SIGMA, HOG_NORM_TYPE, HOG_L2_HYS_THRESH,
    HOG_GAMMA_CORRECTION, HOG_NLEVELS, params['signed']
    )

    hog_dim = hog.getDescriptorSize()
    print(f"HOG feature dimensions: {hog_dim}")

    X_pos = extract_hog_features(pos_images, hog)
    X_neg = extract_hog_features(neg_images_with_hard, hog)
        
    y_pos = np.ones(X_pos.shape[0])
    y_neg = -np.ones(X_neg.shape[0])
    X_train = np.vstack((X_pos, X_neg))
    y_train = np.hstack((y_pos, y_neg))

    svm = LinearSVC(C=SVM_C, max_iter=SVM_MAX_ITER, dual=False, random_state=42)
    svm.fit(X_train, y_train)
    print("SVM training complete！")

    test_scores, test_labels, total_test_windows = get_test_det_results(
        TEST_POS_DIR,
        TEST_ANNO_DIR,
        TEST_NEG_DIR,
        hog,
        svm,
        hog_dim
    )
    print(f"✅ Test set processing complete! Number of valid positive samples：{np.sum(np.array(test_labels)==1)}, Total number of test windows：{total_test_windows}")

    miss_rate, fppw = compute_det_curve(test_scores, test_labels, total_test_windows)
    target_fppw = 1e-4
    closest_idx = np.argmin(np.abs(fppw - target_fppw))
    print(f"在FPPW={fppw[closest_idx]:.2e}时，漏检率={miss_rate[closest_idx]:.2%}")
    
    results_63.append({
        'params': params,
        'test_scores': test_scores,
        'test_labels': test_labels,
        'total_test_windows': total_test_windows,
        'miss_rate': miss_rate,
        'fppw': fppw
    })

print(f"\n📊 A total of {len(results_63)} iterations were recorded")

In [ ]:

colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black', 'orange']

plt.figure(figsize=(12, 8))
for i, result in enumerate(results_63):
    label = result['params']['name']
    plt.plot(result['fppw'], result['miss_rate'], label=label, color=colors[i])

plt.axvline(x=1e-4, color='red', linestyle='--', linewidth=1, label='FPPW=1e-4')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('FPPW')
plt.ylabel('Miss Rate')
plt.title('6.3 HOG with Different Spatial/Orientation Binning Parameters')

plt.ylim(1e-2, 0.5)
plt.yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])

from matplotlib.ticker import FormatStrFormatter
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
with open('results_63.pkl', 'wb') as f:
    pickle.dump(results_63, f)
plt.legend()
plt.grid(True)
plt.show()

### 6.4 Normalization and Descriptor Blocks

In [ ]:
param_combos_64 = [
    {'block_stride': (8, 8), 'l2hys_thresh': 0.2, 'name': 'L2-Hys 50% Overlapping (stride=8)'},
    {'block_stride': (4, 4), 'l2hys_thresh': 0.2, 'name': 'High Overlapping (stride=4)'},
    {'block_stride': (16, 16), 'l2hys_thresh': 0.2, 'name': 'No Overlapping (stride=16)'},
    {'block_stride': (8, 8), 'l2hys_thresh': 1e10, 'name': 'Similar to L2-norm (Large th without clip)'},
]

In [ ]:
results_64 = []

for params in param_combos_64:
    print(f"\n🔧 Test 6.4: Normalisation+overlap: {params['name']}")
    
    hog = cv2.HOGDescriptor(
        WIN_SIZE, HOG_BLOCK_SIZE, params['block_stride'], HOG_CELL_SIZE, HOG_NBINS,
        HOG_DERIV_APERTURE, HOG_WIN_SIGMA, HOG_NORM_TYPE, params['l2hys_thresh'],
        HOG_GAMMA_CORRECTION, HOG_NLEVELS, HOG_SIGNED_GRADIENT
    )

    hog_dim = hog.getDescriptorSize()
    print(f"HOG特征维度: {hog_dim}")

    X_pos = extract_hog_features(pos_images, hog)
    X_neg = extract_hog_features(neg_images_with_hard, hog)
        
    y_pos = np.ones(X_pos.shape[0])
    y_neg = -np.ones(X_neg.shape[0])
    X_train = np.vstack((X_pos, X_neg))
    y_train = np.hstack((y_pos, y_neg))

    svm = LinearSVC(C=SVM_C, max_iter=SVM_MAX_ITER, dual=False, random_state=42)
    svm.fit(X_train, y_train)
    print("SVM training complete!")

    test_scores, test_labels, total_test_windows = get_test_det_results(
        TEST_POS_DIR,
        TEST_ANNO_DIR,
        TEST_NEG_DIR,
        hog,
        svm,
        hog_dim
    )
    print(f"✅ Test set processing complete! Number of valid positive samples：{np.sum(np.array(test_labels)==1)}, Total number of test windows：{total_test_windows}")

    miss_rate, fppw = compute_det_curve(test_scores, test_labels, total_test_windows)
    target_fppw = 1e-4
    closest_idx = np.argmin(np.abs(fppw - target_fppw))
    print(f"When FPPW={fppw[closest_idx]:.2e}，Missed detection rate={miss_rate[closest_idx]:.2%}")
    
    results_64.append({
        'params': params,
        'test_scores': test_scores,
        'test_labels': test_labels,
        'total_test_windows': total_test_windows,
        'miss_rate': miss_rate,
        'fppw': fppw
    })

print(f"\n📊 A total of {len(results_64)} iterations were recorded")

In [ ]:
colors = ['blue', 'green', 'red', 'cyan']

plt.figure(figsize=(12, 8))
for i, result in enumerate(results_64):
    label = result['params']['name']
    plt.plot(result['fppw'], result['miss_rate'], label=label, color=colors[i])

plt.axvline(x=1e-4, color='red', linestyle='--', linewidth=1, label='FPPW=1e-4')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('FPPW')
plt.ylabel('Miss Rate')
plt.title('6.4 HOG with Different Normalization and Overlap Parameters')

plt.ylim(1e-2, 0.5)
plt.yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])

from matplotlib.ticker import FormatStrFormatter
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
with open('results_64.pkl', 'wb') as f:
    pickle.dump(results_64, f)
plt.legend()
plt.grid(True)
plt.show()

### 6.5 Detector Window and Context

In [ ]:
param_combos_65 = [
    {'win_size': (64, 128), 'name': '64x128 (default，16 pixel margin)'},
    {'win_size': (56, 120), 'name': '56x120 (reduce margin)'},
    {'win_size': (48, 112), 'name': '48x112'},
]

#### Temporary function definitions (for use in version 6.5 only)

In [ ]:
def pyramid_65(img, min_size=(64,128)):
    scale = 1.0
    while True:
        scaled = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
        if scaled.shape[0] < min_size[1] or scaled.shape[1] < min_size[0]:
            break
        yield scaled
        scale /= PYRAMID_SCALE

def crop_person_with_margin_65(img, bbox, target_size=(64, 128), aspect_ratio=None):
    if img is None or len(bbox) != 4:
        return None
    
    img_h, img_w = img.shape[:2]
    Xmin, Ymin, Xmax, Ymax = bbox
    
    person_w = Xmax - Xmin
    person_h = Ymax - Ymin
    if person_w <= 0 or person_h <= 0:
        return None

    margin_ratio = 0.25
    margin_w = int(person_w * margin_ratio)
    margin_h = int(person_h * margin_ratio)

    new_x1 = max(0, Xmin - margin_w)
    new_y1 = max(0, Ymin - margin_h)
    new_x2 = min(img_w, Xmax + margin_w)
    new_y2 = min(img_h, Ymax + margin_h)
    
    crop_w = new_x2 - new_x1
    crop_h = new_y2 - new_y1

    if aspect_ratio is None:
        aspect_ratio = target_size[0] / target_size[1]  
    
    current_ratio = crop_w / crop_h if crop_h > 0 else 1.0
    
    if current_ratio > aspect_ratio:
        target_h = int(crop_w / aspect_ratio)
        delta_h = target_h - crop_h
        extend_top = delta_h // 2
        extend_bottom = delta_h - extend_top
        
        new_y1 = max(0, new_y1 - extend_top)
        new_y2 = min(img_h, new_y2 + extend_bottom)
        crop_h = new_y2 - new_y1
        crop_w = new_x2 - new_x1
    else:
        target_w = int(crop_h * aspect_ratio)
        delta_w = target_w - crop_w
        extend_left = delta_w // 2
        extend_right = delta_w - extend_left
        
        new_x1 = max(0, new_x1 - extend_left)
        new_x2 = min(img_w, new_x2 + extend_right)
        crop_w = new_x2 - new_x1
        crop_h = new_y2 - new_y1
    
    cropped = img[new_y1:new_y2, new_x1:new_x2]
    
    if cropped.size == 0:
        return None
    
    try:
        resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
        return resized
    except Exception as e:
        print(f"resize fail: {str(e)}")
        return None

def get_test_det_results_win(pos_img_dir, anno_dir, neg_dir, hog_detector, svm_model, hog_dim, win_size=(64, 128)):
    WIN_W, WIN_H = win_size
    all_scores = []
    all_labels = []
    total_test_windows = 0
    print("📥 Processing test positive samples (analysing, annotating and cropping pedestrians)...")
    pos_img_list = [f for f in os.listdir(pos_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(pos_img_list):
        img_path = os.path.join(pos_img_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        
        img_prefix = os.path.splitext(img_name)[0]
        anno_path = os.path.join(anno_dir, f"{img_prefix}.txt")
        if not os.path.exists(anno_path):
            continue
        
        bbox_list = parse_inria_annotation(anno_path)
        if not bbox_list:
            continue
        
        for bbox in bbox_list:
            try:
                person_img = crop_person_with_margin_65(img, bbox, target_size=win_size, aspect_ratio=win_size[0]/win_size[1])  
                if person_img is not None and person_img.shape[:2] == (win_size[1], win_size[0]):
                    feature = hog_detector.compute(person_img).flatten().reshape(1, -1)
                    score = svm_model.decision_function(feature)[0] + 1.5
                    all_scores.append(score)
                    all_labels.append(1)

            except Exception as e:
                print(f"Failure to process the master copy: {img_name}, {e}")
                continue


    print("📥 Processing negative test samples (exhaustive sliding window search)...")
    neg_img_list = [f for f in os.listdir(neg_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(neg_img_list):
        img_path = os.path.join(neg_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        
        for scale_img in pyramid_65(img): 
            img_h, img_w = scale_img.shape
            
            x_coords = np.arange(0, img_w - WIN_W + 1, WINDOW_STRIDE)
            y_coords = np.arange(0, img_h - WIN_H + 1, WINDOW_STRIDE)
            
            if len(x_coords) == 0 or len(y_coords) == 0:
                continue
            
            num_windows = len(x_coords) * len(y_coords)
            total_test_windows += num_windows
            
            batch_features = np.zeros((num_windows, hog_dim), dtype=np.float32)
            window_idx = 0
            for y in y_coords:
                for x in x_coords:
                    window = scale_img[y:y+WIN_H, x:x+WIN_W]
                    batch_features[window_idx] = hog_detector.compute(window).flatten()
                    window_idx += 1
            
            batch_scores = svm_model.decision_function(batch_features)
            all_scores.extend(batch_scores)
            all_labels.extend([-1] * len(batch_scores))

    return np.array(all_scores), np.array(all_labels), total_test_windows

#### Training process

In [ ]:
results_65 = []

for params in param_combos_65:
    print(f"\n🔧 Test 6.5: Detection window: {params['name']}")
    WIN_SIZE_TEMP = params['win_size']  
    
    hog = cv2.HOGDescriptor(
        WIN_SIZE_TEMP, HOG_BLOCK_SIZE, HOG_BLOCK_STRIDE, HOG_CELL_SIZE, HOG_NBINS,
        HOG_DERIV_APERTURE, HOG_WIN_SIGMA, HOG_NORM_TYPE, HOG_L2_HYS_THRESH,
        HOG_GAMMA_CORRECTION, HOG_NLEVELS, HOG_SIGNED_GRADIENT
    )
    
    hog_dim = hog.getDescriptorSize()
    print(f"HOG特征维度: {hog_dim}")

    print("正在重新提取特征（resize + compute）...")
    X_pos = []
    for img in tqdm(pos_images, desc="正样本"):
        if img.shape[:2] != (WIN_SIZE_TEMP[1], WIN_SIZE_TEMP[0]):
            img = cv2.resize(img, WIN_SIZE_TEMP, interpolation=cv2.INTER_LINEAR)
        X_pos.append(hog.compute(img).flatten())

    X_neg = []
    for img in tqdm(neg_images_with_hard, desc="负样本"):
        if img.shape[:2] != (WIN_SIZE_TEMP[1], WIN_SIZE_TEMP[0]):
            img = cv2.resize(img, WIN_SIZE_TEMP, interpolation=cv2.INTER_LINEAR)
        X_neg.append(hog.compute(img).flatten())

    X_pos = np.array(X_pos)
    X_neg = np.array(X_neg)
        
    y_pos = np.ones(X_pos.shape[0])
    y_neg = -np.ones(X_neg.shape[0])
    X_train = np.vstack((X_pos, X_neg))
    y_train = np.hstack((y_pos, y_neg))

    svm = LinearSVC(C=SVM_C, max_iter=SVM_MAX_ITER, dual=False, random_state=42)
    svm.fit(X_train, y_train)
    print("SVM训练完成！")

    test_scores, test_labels, total_test_windows = get_test_det_results_win(
        TEST_POS_DIR,
        TEST_ANNO_DIR,
        TEST_NEG_DIR,
        hog,
        svm,
        hog_dim,
        win_size=WIN_SIZE_TEMP
    )
    print(f"✅ Test set processing complete! Number of valid positive samples：{np.sum(np.array(test_labels)==1)}, Total number of test windows：{total_test_windows}")

    miss_rate, fppw = compute_det_curve(test_scores, test_labels, total_test_windows)
    target_fppw = 1e-4
    closest_idx = np.argmin(np.abs(fppw - target_fppw))
    print(f"When FPPW={fppw[closest_idx]:.2e}，Missed detection rate={miss_rate[closest_idx]:.2%}")
    
    results_65.append({
        'params': params,
        'test_scores': test_scores,
        'test_labels': test_labels,
        'total_test_windows': total_test_windows,
        'miss_rate': miss_rate,
        'fppw': fppw
    })

print(f"\n📊 A total of {len(results_65)} iterations were recorded")

In [ ]:
colors = ['blue', 'green', 'red']

plt.figure(figsize=(12, 8))
for i, result in enumerate(results_65):
    label = result['params']['name']
    plt.plot(result['fppw'], result['miss_rate'], label=label, color=colors[i])

plt.axvline(x=1e-4, color='red', linestyle='--', linewidth=1, label='FPPW=1e-4')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('FPPW')
plt.ylabel('Miss Rate')
plt.title('6.5 HOG with Different Detection Window Sizes')

plt.ylim(1e-2, 0.5)
plt.yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])

from matplotlib.ticker import FormatStrFormatter
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
with open('results_65.pkl', 'wb') as f:
    pickle.dump(results_65, f)
plt.legend()
plt.grid(True)
plt.show()

### 6.6 Linear vs Kernel SVM

In [ ]:
param_combos_66 = [
    {'kernel': 'linear', 'gamma': None, 'name': 'Linear SVM (默认)'},
    {'kernel': 'rbf', 'gamma': 0.008, 'name': 'Kernel γ=0.008'},
    {'kernel': 'rbf', 'gamma': 0.03, 'name': 'Kernel γ=0.03'},
    {'kernel': 'rbf', 'gamma': 0.07, 'name': 'Kernel γ=0.07'},
]

#### Temporary function definitions (for use in 6.6 only)

In [ ]:
def get_test_det_results_classifier(pos_img_dir, anno_dir, neg_dir, hog_detector, svm_model):
    hog_dim = hog_detector.getDescriptorSize()   
    
    all_scores = []
    all_labels = []
    total_test_windows = 0

    print("📥 Processing test positive samples (parsing, labelling and cropping pedestrians)...")
    pos_img_list = [f for f in os.listdir(pos_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(pos_img_list):
        img_path = os.path.join(pos_img_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        
        img_prefix = os.path.splitext(img_name)[0]
        anno_path = os.path.join(anno_dir, f"{img_prefix}.txt")
        if not os.path.exists(anno_path):
            continue
        
        bbox_list = parse_inria_annotation(anno_path)
        if len(bbox_list) == 0:
            continue
        
        for bbox in bbox_list:
            try:
                person_img = crop_person_with_margin(img, bbox)
                feature = hog_detector.compute(person_img).flatten().reshape(1, -1)
                score = svm_model.decision_function(feature)[0] + 1.5
                all_scores.append(score)
                all_labels.append(1)
            except Exception as e:
                continue

    print("📥 Processing negative test samples (exhaustive sliding window search)...")
    neg_img_list = [f for f in os.listdir(neg_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.ppm'))]
    
    for img_name in tqdm(neg_img_list):
        img_path = os.path.join(neg_dir, img_name)
        img = safe_read_image(img_path, to_gray=True)
        if img is None:
            continue
        
        for scale_img in pyramid(img):
            img_h, img_w = scale_img.shape
            x_coords = np.arange(0, img_w - WIN_W + 1, WINDOW_STRIDE)
            y_coords = np.arange(0, img_h - WIN_H + 1, WINDOW_STRIDE)
            
            if len(x_coords) == 0 or len(y_coords) == 0:
                continue
            
            num_windows = len(x_coords) * len(y_coords)
            total_test_windows += num_windows
            
            batch_features = np.zeros((num_windows, hog_dim), dtype=np.float32)  
            window_idx = 0
            for y in y_coords:
                for x in x_coords:
                    window = scale_img[y:y+WIN_H, x:x+WIN_W]
                    batch_features[window_idx] = hog_detector.compute(window).flatten()
                    window_idx += 1
            
            batch_scores = svm_model.decision_function(batch_features)
            all_scores.extend(batch_scores)
            all_labels.extend([-1] * len(batch_scores))

    return np.array(all_scores), np.array(all_labels), total_test_windows

#### Training process

In [ ]:
results_66 = []

for params in param_combos_66:
    print(f"\n🔧 Test 6.6 Classifier: {params['name']}")
    
    if params['kernel'] == 'linear':
        svm = LinearSVC(C=SVM_C, max_iter=SVM_MAX_ITER, dual=False, random_state=42)
    else:
        svm = SVC(kernel=params['kernel'], gamma=params['gamma'], C=SVM_C, random_state=42)
    
    hog = cv2.HOGDescriptor(
        WIN_SIZE, HOG_BLOCK_SIZE, HOG_BLOCK_STRIDE, HOG_CELL_SIZE, HOG_NBINS,
        HOG_DERIV_APERTURE, HOG_WIN_SIGMA, HOG_NORM_TYPE, HOG_L2_HYS_THRESH,
        HOG_GAMMA_CORRECTION, HOG_NLEVELS, HOG_SIGNED_GRADIENT
    )

    X_pos = extract_hog_features(pos_images, hog)
    X_neg = extract_hog_features(neg_images_with_hard, hog)
    
    y_pos = np.ones(X_pos.shape[0])
    y_neg = -np.ones(X_neg.shape[0])
    X_train = np.vstack((X_pos, X_neg))
    y_train = np.hstack((y_pos, y_neg))

    svm.fit(X_train, y_train)
    print("SVM training complete！")
    
    test_scores, test_labels, total_test_windows = get_test_det_results_classifier(
        TEST_POS_DIR, TEST_ANNO_DIR, TEST_NEG_DIR, hog, svm
    )
    print(f"✅ Test complete! Total number of windows：{total_test_windows}")
    
    miss_rate, fppw = compute_det_curve(test_scores, test_labels, total_test_windows)
    closest_idx = np.argmin(np.abs(fppw - 1e-4))
    print(f"The false negative rate when FPPW ≈ 1e-4: {miss_rate[closest_idx]:.2%}")
    
    results_66.append({
        'params': params,
        'miss_rate': miss_rate,
        'fppw': fppw
    })

print(f"\n📊 A total of {len(results_66)} iterations were recorded")

In [ ]:
colors = ['blue', 'green', 'red', 'cyan']

plt.figure(figsize=(12, 8))
for i, result in enumerate(results_66):
    label = result['params']['name']
    plt.plot(result['fppw'], result['miss_rate'], label=label, color=colors[i])

plt.axvline(x=1e-4, color='red', linestyle='--', linewidth=1, label='FPPW=1e-4')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('FPPW')
plt.ylabel('Miss Rate')
plt.title('6.6 HOG with Different Classifiers')

plt.ylim(1e-2, 0.5)
plt.yticks([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])

from matplotlib.ticker import FormatStrFormatter
plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
with open('results_66.pkl', 'wb') as f:
    pickle.dump(results_66, f)
plt.legend()
plt.grid(True)
plt.show()